# AI4T Mini-Project - Group 2
## Cross-Domain Trust in 4G/5G YouTube QoE Prediction

**This is the ONLY notebook. Run every cell in order, top to bottom.**
Total runtime: about 12 minutes.

It produces all five required code components, all result tables,
and all six figures.


## 1. Dataset  *(~15 s)*

In [1]:
!git clone https://github.com/razaulmustafa852/youtubegoes5g.git yt5g
!ls yt5g

Cloning into 'yt5g'...
remote: Enumerating objects: 894, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 894 (delta 20), reused 93 (delta 12), pack-reused 782 (from 1)
Receiving objects: 100% (894/894), 6.24 MiB | 6.94 MiB/s, done.
Resolving deltas: 100% (327/327), done.
'Channel Logs'	 Models   README.md  'YouTube QoE'  'YouTuve QoE Events'


## 2. Component 1/5 - chronological replay, twin state, labels

In [2]:
%%writefile twin_replay.py
"""
AI4T Project 2 -- Cross-Domain Trust in 5G/4G YouTube QoE
Component 1 of 5: CHRONOLOGICAL REPLAY + TWIN STATE + LABEL DEFINITION

This module is the "digital twin" core required by the brief:
  (1) it replays observations chronologically, one session at a time,
      never treating rows as independent shuffled samples;
  (2) it maintains a STATE holding the current network/service variables;
  (3) it emits a label for a FUTURE event (stall within the next H seconds).

Dataset: razaulmustafa852/youtubegoes5g
  Channel Logs/<Context>/<Eid>.csv  -- 1 Hz radio + throughput telemetry
  YouTuve QoE Events/events.csv     -- YouTube IFrame player state changes

Author: <group name>
"""

from __future__ import annotations

import csv
import glob
import os
import re
from collections import Counter, deque
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Iterator

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------

#  Where the cloned dataset lives. By default we look for a folder named
#  "yt5g" sitting next to this script. Override with the AI4T_DATA env var,
#  e.g.  export AI4T_DATA=/Users/you/Downloads/youtubegoes5g
_HERE = os.path.dirname(os.path.abspath(__file__))
DATA_ROOT = os.environ.get("AI4T_DATA", os.path.join(_HERE, "yt5g"))

#  Prediction horizon. The telemetry is sampled at 1 Hz, so a 5 s horizon is
#  5 samples ahead. The brief for Project 2 explicitly names "a stall during
#  the next five seconds" as an admissible target, so H = 5.
HORIZON_S = 5

#  Length of the history window used to build rolling features. Kept short so
#  that a prediction can be issued only 10 s into a session.
WINDOW_S = 10

#  Radio / service columns we lift out of the raw channel logs.
#  NOTE on naming: in this dataset 'Level' is RSRP (dBm) and 'Qual' is RSRQ (dB).
#  We rename them so the report and the LLM evidence use standard 3GPP terms.
RAW_TO_STD = {
    "Level": "rsrp",
    "Qual": "rsrq",
    "SNR": "snr",
    "CQI": "cqi",
    "DL_bitrate": "dl_bitrate",
    "UL_bitrate": "ul_bitrate",
    "SecondCell_RSRP": "sc_rsrp",
    "SecondCell_RSRQ": "sc_rsrq",
    "SecondCell_SNR": "sc_snr",
}

NUMERIC_COLS = list(RAW_TO_STD.values())

#  Physically plausible ranges. Rows outside these are treated as corrupt.
#  ~1-2% of rows in this dataset are column-misaligned (numeric junk lands in
#  NetworkTech), and this is how we catch them without hand-editing the data.
VALID_RANGE = {
    "rsrp": (-140.0, -40.0),      # dBm
    "rsrq": (-30.0, 0.0),         # dB
    "snr": (-20.0, 40.0),         # dB
    "cqi": (0.0, 30.0),           # index
    "dl_bitrate": (0.0, 2_000_000.0),   # kbps as logged
    "ul_bitrate": (0.0, 2_000_000.0),
    "sc_rsrp": (-140.0, -40.0),
    "sc_rsrq": (-30.0, 0.0),
    "sc_snr": (-20.0, 40.0),
}


# --------------------------------------------------------------------------
# Low-level parsing helpers
# --------------------------------------------------------------------------

def _norm_eid(s: str) -> str:
    """Session ids are inconsistently cased between the two files
    ('4P7s2' in the log filename vs '4p7s2' in events.csv). Normalising is
    what takes the join from 3 sessions to 264."""
    return (s or "").strip().lower()


def _to_float(x) -> float:
    """Robust numeric cast. The logs use '-' for 'not measured'."""
    if x is None:
        return np.nan
    s = str(x).strip().strip('"')
    if s in ("", "-", "nan", "None", "NA"):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan


def _parse_log_ts(x) -> datetime | None:
    """Channel-log timestamps look like '"\n2022.09.26_17.42.11"'."""
    if x is None:
        return None
    s = re.sub(r"[\r\n\"]", "", str(x)).strip()
    try:
        return datetime.strptime(s, "%Y.%m.%d_%H.%M.%S")
    except ValueError:
        return None


def _parse_event_ts(date_s: str, time_s: str) -> datetime | None:
    """events.csv splits the stamp into Date (dd/mm/YYYY) and Time (HH:MM:SS)."""
    try:
        return datetime.strptime(
            f"{str(date_s).strip()} {str(time_s).strip()}", "%d/%m/%Y %H:%M:%S"
        )
    except (ValueError, TypeError):
        return None


# --------------------------------------------------------------------------
# Loading
# --------------------------------------------------------------------------

@dataclass
class Session:
    """One YouTube streaming session: telemetry + player events + domain tags."""
    eid: str
    context: str                 # Indoor | Outdoor | Pedestrian | Mobility
    tech: str                    # 4G | 5G  (dominant technology in the session)
    telemetry: pd.DataFrame      # 1 Hz, chronologically sorted, indexed by second
    stall_starts: list           # datetimes at which a rebuffering event begins
    playback_start: datetime | None  # first 'playing' event; startup buffering
                                     # before this is NOT counted as a stall
    n_dropped_rows: int = 0      # corrupt rows removed, reported in the paper


def load_events(root: str = DATA_ROOT) -> dict:
    """Read the player-event log and group it by (normalised) session id."""
    path = os.path.join(root, "YouTuve QoE Events", "events.csv")
    by_eid: dict[str, list] = {}
    with open(path, encoding="utf-8", errors="replace") as fh:
        for row in csv.DictReader(fh):
            ts = _parse_event_ts(row.get("Date"), row.get("Time"))
            if ts is None:
                continue
            by_eid.setdefault(_norm_eid(row.get("Eid")), []).append(
                {
                    "ts": ts,
                    "category": (row.get("Category") or "").strip(),
                    "quality": (row.get("Quality") or "").strip().rstrip(","),
                    "time_stall": _to_float(row.get("TimeStall")),
                }
            )
    for eid in by_eid:
        by_eid[eid].sort(key=lambda r: r["ts"])
    return by_eid


def _clean_telemetry(rows: list) -> tuple[pd.DataFrame, int]:
    """Turn raw log rows into a clean, chronologically sorted 1 Hz frame.

    Returns the frame and the number of rows discarded, so the report can
    state the data-quality cost honestly rather than hiding it.
    """
    recs, dropped = [], 0
    for r in rows:
        ts = _parse_log_ts(r.get("Timestamp"))
        tech = (r.get("NetworkTech") or "").strip()
        # Column-misalignment guard: NetworkTech must be a technology string.
        if ts is None or tech not in ("2G", "3G", "4G", "5G"):
            dropped += 1
            continue
        rec = {"ts": ts, "tech": tech,
               "event": (r.get("EVENT") or "").strip(),
               "state": (r.get("State") or "").strip()}
        for raw, std in RAW_TO_STD.items():
            rec[std] = _to_float(r.get(raw))
        recs.append(rec)

    if not recs:
        return pd.DataFrame(), dropped

    df = pd.DataFrame(recs).sort_values("ts").reset_index(drop=True)

    # Range-check each physical quantity; out-of-range becomes NaN rather than
    # a silently wrong feature value.
    for col, (lo, hi) in VALID_RANGE.items():
        if col in df.columns:
            df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan

    # Collapse duplicate timestamps (the logger occasionally double-writes).
    df = df.drop_duplicates(subset="ts", keep="first").reset_index(drop=True)
    return df, dropped


def load_sessions(root: str = DATA_ROOT, verbose: bool = True) -> list[Session]:
    """Load every session for which BOTH telemetry and player events exist."""
    events = load_events(root)
    sessions, skipped = [], Counter()

    for path in sorted(glob.glob(os.path.join(root, "Channel Logs", "*", "*.csv"))):
        eid = _norm_eid(os.path.splitext(os.path.basename(path))[0])
        context = os.path.basename(os.path.dirname(path))

        if eid not in events:
            skipped["no_player_events"] += 1
            continue

        with open(path, encoding="utf-8", errors="replace") as fh:
            raw_rows = list(csv.DictReader(fh))
        df, dropped = _clean_telemetry(raw_rows)

        if len(df) < WINDOW_S + HORIZON_S + 5:
            skipped["too_short"] += 1
            continue

        ev = events[eid]
        playing = [e["ts"] for e in ev if e["category"] == "playing"]
        playback_start = min(playing) if playing else None

        # A rebuffering event that happens *after* playback has begun is a
        # stall. Buffering before the first 'playing' event is startup delay,
        # a different QoE impairment, and is excluded from the target.
        stalls = [
            e["ts"] for e in ev
            if e["category"] == "buffering"
            and playback_start is not None
            and e["ts"] > playback_start
        ]

        tech_counts = Counter(df["tech"])
        tech = tech_counts.most_common(1)[0][0]
        if tech not in ("4G", "5G"):
            skipped["odd_tech"] += 1
            continue

        sessions.append(
            Session(eid=eid, context=context, tech=tech, telemetry=df,
                    stall_starts=sorted(stalls), playback_start=playback_start,
                    n_dropped_rows=dropped)
        )

    if verbose:
        print(f"loaded {len(sessions)} sessions   skipped: {dict(skipped)}")
    return sessions


# --------------------------------------------------------------------------
# The twin state
# --------------------------------------------------------------------------

@dataclass
class TwinState:
    """The state of the service twin at one instant of replay.

    This is deliberately a real object rather than a row of a feature matrix:
    it is updated incrementally as the replay advances, it only ever contains
    information observable at or before time t, and it is what gets serialised
    into the evidence dictionary handed to the LLM.
    """
    eid: str
    context: str
    tech: str
    t: datetime | None = None
    elapsed_s: float = 0.0

    # instantaneous radio + service variables
    current: dict = field(default_factory=dict)
    # rolling history, most recent last
    history: deque = field(default_factory=lambda: deque(maxlen=WINDOW_S))
    # cumulative session counters
    n_handovers: int = 0
    n_stalls_so_far: int = 0

    def update(self, row: pd.Series) -> None:
        """Advance the twin by one observation."""
        self.t = row["ts"]
        self.current = {c: row.get(c, np.nan) for c in NUMERIC_COLS}
        self.current["tech_is_5g"] = 1.0 if row["tech"] == "5G" else 0.0
        self.history.append(dict(self.current))
        if "HANDOVER" in str(row.get("event", "")):
            self.n_handovers += 1

    def features(self) -> dict | None:
        """Feature vector built ONLY from the window ending at t.

        Every quantity here is computable by an operator at time t. Nothing
        reads forward. This is the single most important property of the
        pipeline -- a leaked future value would inflate every metric in the
        report.
        """
        if len(self.history) < WINDOW_S:
            return None   # not enough history yet; twin stays in warm-up

        hist = pd.DataFrame(list(self.history))
        f: dict = {}

        for c in ["rsrp", "rsrq", "snr", "cqi", "dl_bitrate"]:
            s = hist[c]
            f[f"{c}_last"] = s.iloc[-1]
            f[f"{c}_mean"] = s.mean()
            f[f"{c}_std"] = s.std()
            f[f"{c}_min"] = s.min()
            f[f"{c}_max"] = s.max()
            # short-term trend: is the channel improving or collapsing?
            f[f"{c}_delta"] = s.iloc[-1] - s.iloc[0]
            f[f"{c}_slope"] = _slope(s.values)

        # secondary (NR/LTE dual-connectivity) cell, when reported
        for c in ["sc_rsrp", "sc_rsrq", "sc_snr"]:
            f[f"{c}_mean"] = hist[c].mean()

        # throughput collapse indicators -- the strongest physical precursor
        # of a rebuffering event
        dl = hist["dl_bitrate"]
        f["dl_zero_frac"] = float((dl.fillna(0) <= 0).mean())
        f["dl_ratio_last_to_mean"] = (
            dl.iloc[-1] / dl.mean() if dl.mean() and dl.mean() > 0 else np.nan
        )

        f["tech_is_5g"] = self.current.get("tech_is_5g", np.nan)
        f["n_handovers"] = float(self.n_handovers)
        f["n_stalls_so_far"] = float(self.n_stalls_so_far)
        f["elapsed_s"] = float(self.elapsed_s)
        return f


def _slope(v: np.ndarray) -> float:
    """Least-squares slope of a short series; NaN-safe."""
    v = np.asarray(v, dtype=float)
    m = ~np.isnan(v)
    if m.sum() < 2:
        return np.nan
    x = np.arange(len(v))[m]
    return float(np.polyfit(x, v[m], 1)[0])


# --------------------------------------------------------------------------
# Replay
# --------------------------------------------------------------------------

def replay_session(sess: Session, horizon_s: int = HORIZON_S) -> pd.DataFrame:
    """Replay ONE session second by second and emit (state, label) pairs.

    The label is the required future event:
        y = 1  iff a rebuffering event starts in the interval (t, t + H]

    Rows before the twin has warmed up, and rows in the final H seconds
    (whose future is not observable), are not emitted.
    """
    df = sess.telemetry
    state = TwinState(eid=sess.eid, context=sess.context, tech=sess.tech)
    stalls = np.array([s.timestamp() for s in sess.stall_starts])
    t_end = df["ts"].iloc[-1]
    t0 = df["ts"].iloc[0]

    out = []
    for _, row in df.iterrows():          # <-- chronological, never shuffled
        state.elapsed_s = (row["ts"] - t0).total_seconds()
        state.update(row)

        # count stalls already seen, so the state knows session history
        state.n_stalls_so_far = int((stalls <= row["ts"].timestamp()).sum())

        feats = state.features()
        if feats is None:
            continue
        # the last H seconds have no observable future -> cannot be labelled
        if (t_end - row["ts"]).total_seconds() < horizon_s:
            continue

        lo = row["ts"].timestamp()
        hi = (row["ts"] + timedelta(seconds=horizon_s)).timestamp()
        y = int(((stalls > lo) & (stalls <= hi)).any())

        feats.update({
            "eid": sess.eid,
            "context": sess.context,
            "tech": sess.tech,
            "ts": row["ts"],
            "y": y,
        })
        out.append(feats)

    return pd.DataFrame(out)


def build_dataset(sessions: list[Session], horizon_s: int = HORIZON_S,
                  verbose: bool = True) -> pd.DataFrame:
    """Replay every session and concatenate. Session order is preserved."""
    frames = [replay_session(s, horizon_s) for s in sessions]
    frames = [f for f in frames if len(f)]
    data = pd.concat(frames, ignore_index=True)
    if verbose:
        print(f"replayed {len(frames)} sessions -> {len(data):,} labelled instants")
        print(f"positive rate: {data['y'].mean():.4f}  "
              f"({int(data['y'].sum()):,} stall-imminent instants)")
    return data


if __name__ == "__main__":
    if not os.path.isdir(DATA_ROOT):
        raise SystemExit(
            f"\nCannot find the dataset at: {DATA_ROOT}\n"
            "Clone it first:\n"
            "    git clone https://github.com/razaulmustafa852/youtubegoes5g.git yt5g\n"
            "or point AI4T_DATA at wherever you put it.\n"
        )
    sessions = load_sessions()
    data = build_dataset(sessions)
    out = os.path.join(_HERE, "replayed.pkl")
    data.to_pickle(out)
    print(f"\nsaved -> {out}")
    print("\nstall rate by domain:")
    print(data.groupby(["tech", "context"])["y"].agg(["size", "mean"]))


Writing twin_replay.py


## 3. Component 2/5 - baselines and models

In [3]:
%%writefile models.py
"""
AI4T Project 2 -- Cross-Domain Trust in 4G/5G YouTube QoE
Component 2 of 5: BASELINES, MODELS, AND DOMAIN-AWARE EVALUATION

Answers the parts of the brief that require:
  - a comparison of Logistic Regression, Random Forest and Gradient Boosting
  - in-domain versus out-of-domain F1
  - the generalization gap
  - calibration degradation across domains

CRITICAL METHODOLOGICAL POINT
-----------------------------
Splits are made by SESSION, never by row. Second 40 and second 41 of the same
streaming session share almost all of their feature values, so a random row
split places near-duplicates on both sides of the train/test boundary and
produces a badly inflated score. We demonstrate this explicitly in
`leakage_demonstration()` because it is a result worth reporting, not just a
mistake worth avoiding.
"""

from __future__ import annotations

import os
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

_HERE = os.path.dirname(os.path.abspath(__file__))
RANDOM_STATE = 42
HORIZON_S = 5

# Columns that describe the instance rather than the network state.
META_COLS = ["eid", "context", "tech", "ts", "y"]

# The four measurement contexts split into a mobility axis.
STATIC_CONTEXTS = {"Indoor", "Outdoor"}
MOBILE_CONTEXTS = {"Pedestrian", "Mobility"}


# --------------------------------------------------------------------------
# Data preparation
# --------------------------------------------------------------------------

def load_replayed(path: str | None = None) -> pd.DataFrame:
    path = path or os.path.join(_HERE, "replayed.pkl")
    df = pd.read_pickle(path)
    df = df.sort_values(["eid", "ts"]).reset_index(drop=True)
    df["mobility"] = np.where(df["context"].isin(STATIC_CONTEXTS), "static", "mobile")
    return df


def feature_columns(df: pd.DataFrame) -> list[str]:
    """Everything that is not metadata. All of these are computed from the
    trailing window only, so none of them can see the future."""
    return [c for c in df.columns if c not in META_COLS + ["mobility"]]


def add_persistence(df: pd.DataFrame, horizon_s: int = HORIZON_S) -> pd.DataFrame:
    """The naive baseline required by the brief.

    Persistence says: 'the near future looks like the recent past'. Concretely,
    predict a stall in (t, t+H] if a stall actually began in (t-H, t].

    That past fact is exactly y evaluated at time t-H, so we obtain it with a
    time-aligned self-join. This uses only information an operator already had
    at time t -- it is a baseline, not a leak.
    """
    left = df[["eid", "ts", "y"]].copy()
    right = df[["eid", "ts", "y"]].copy()
    right["ts"] = right["ts"] + pd.Timedelta(seconds=horizon_s)
    right = right.rename(columns={"y": "persistence"})
    merged = left.merge(right[["eid", "ts", "persistence"]], on=["eid", "ts"], how="left")
    df = df.copy()
    df["persistence"] = merged["persistence"].fillna(0).astype(int).values
    return df


# --------------------------------------------------------------------------
# Metrics
# --------------------------------------------------------------------------

def expected_calibration_error(y_true, y_prob, n_bins: int = 10) -> float:
    """ECE: average gap between predicted confidence and observed frequency.

    A model can have excellent F1 and a terrible ECE. That divergence is the
    central claim of this project, so ECE is reported everywhere alongside F1.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (y_prob > lo) & (y_prob <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / len(y_prob)) * abs(y_true[m].mean() - y_prob[m].mean())
    return float(ece)


def evaluate(y_true, y_prob, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true)
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    pos = int(y_true.sum())
    return {
        "n": len(y_true),
        "n_pos": pos,
        "base_rate": float(y_true.mean()),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_prob) if pos else np.nan,
        "roc_auc": roc_auc_score(y_true, y_prob) if 0 < pos < len(y_true) else np.nan,
        "brier": brier_score_loss(y_true, y_prob),
        "ece": expected_calibration_error(y_true, y_prob),
    }


# --------------------------------------------------------------------------
# Models
# --------------------------------------------------------------------------

def build_models() -> dict:
    """The three models named in the brief.

    class_weight='balanced' matters here: the positive class is 2.6% of rows,
    so an unweighted model can reach 97% accuracy by never predicting a stall.
    """
    imp = lambda: SimpleImputer(strategy="median")
    return {
        "LogisticRegression": Pipeline([
            ("imp", imp()),
            ("sc", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                       random_state=RANDOM_STATE)),
        ]),
        "RandomForest": Pipeline([
            ("imp", imp()),
            ("clf", RandomForestClassifier(n_estimators=100, min_samples_leaf=20,
                                           class_weight="balanced_subsample",
                                           n_jobs=2, random_state=RANDOM_STATE)),
        ]),
        # HistGradientBoosting is the histogram-based implementation. It is
        # 10-50x faster than GradientBoostingClassifier at this data size and
        # handles NaN natively. Naive GradientBoosting exhausted our machine.
        "GradientBoosting": HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.1, max_depth=6,
            class_weight="balanced", random_state=RANDOM_STATE),
    }


def fit_predict(model, Xtr, ytr, Xte):
    model.fit(Xtr, ytr)
    return model.predict_proba(Xte)[:, 1]


# --------------------------------------------------------------------------
# Experiment 1 -- why the split rule matters
# --------------------------------------------------------------------------

def leakage_demonstration(df: pd.DataFrame, feats: list[str]) -> pd.DataFrame:
    """Same model, same data, two split rules. Reported as a figure."""
    X, y, g = df[feats].values, df["y"].values, df["eid"].values
    rows = []

    # (a) naive random row split -- what a leaky pipeline would do
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.permutation(len(df))
    cut = int(0.7 * len(df))
    tr, te = idx[:cut], idx[cut:]
    p = fit_predict(build_models()["RandomForest"], X[tr], y[tr], X[te])
    rows.append({"split": "random row split (leaky)", **evaluate(y[te], p)})

    # (b) grouped split -- sessions never appear on both sides
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_STATE)
    tr, te = next(gss.split(X, y, groups=g))
    p = fit_predict(build_models()["RandomForest"], X[tr], y[tr], X[te])
    rows.append({"split": "session-level split (correct)", **evaluate(y[te], p)})

    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# Experiment 2 -- in-domain versus out-of-domain
# --------------------------------------------------------------------------

def domain_experiment(df: pd.DataFrame, feats: list[str], column: str,
                      source: str, target: str) -> pd.DataFrame:
    """Train on `source`, evaluate in-domain and on `target`.

    In-domain testing still uses a held-out set of SESSIONS from the source
    domain, so the in-domain and out-of-domain numbers are comparable: neither
    has ever seen its test sessions during training.
    """
    src = df[df[column] == source]
    tgt = df[df[column] == target]

    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_STATE)
    tr_i, te_i = next(gss.split(src[feats].values, src["y"].values,
                                groups=src["eid"].values))
    Xtr, ytr = src[feats].values[tr_i].astype("float32"), src["y"].values[tr_i]
    Xin, yin = src[feats].values[te_i].astype("float32"), src["y"].values[te_i]
    Xout, yout = tgt[feats].values.astype("float32"), tgt["y"].values

    rows = []

    # persistence baseline, scored on the same test sets
    rows.append({"model": "Persistence", "condition": f"in-domain ({source})",
                 **evaluate(yin, src["persistence"].values[te_i])})
    rows.append({"model": "Persistence", "condition": f"out-of-domain ({target})",
                 **evaluate(yout, tgt["persistence"].values)})

    for name, model in build_models().items():
        model.fit(Xtr, ytr)
        p_in = model.predict_proba(Xin)[:, 1]
        p_out = model.predict_proba(Xout)[:, 1]
        rows.append({"model": name, "condition": f"in-domain ({source})",
                     **evaluate(yin, p_in)})
        rows.append({"model": name, "condition": f"out-of-domain ({target})",
                     **evaluate(yout, p_out)})

    out = pd.DataFrame(rows)
    out["experiment"] = f"{source} -> {target}"
    return out


def generalization_gap(res: pd.DataFrame) -> pd.DataFrame:
    """F1 gap and calibration degradation, per model, per experiment."""
    rows = []
    for (exp, model), g in res.groupby(["experiment", "model"]):
        ind = g[g["condition"].str.startswith("in-domain")].iloc[0]
        ood = g[g["condition"].str.startswith("out-of-domain")].iloc[0]
        rows.append({
            "experiment": exp, "model": model,
            "f1_in": ind["f1"], "f1_out": ood["f1"],
            "f1_gap": ind["f1"] - ood["f1"],
            "ece_in": ind["ece"], "ece_out": ood["ece"],
            "ece_degradation": ood["ece"] - ind["ece"],
            "brier_in": ind["brier"], "brier_out": ood["brier"],
            "base_rate_in": ind["base_rate"], "base_rate_out": ood["base_rate"],
        })
    return pd.DataFrame(rows).sort_values(["experiment", "model"])


# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------

def main():
    df = load_replayed()
    df = add_persistence(df)
    feats = feature_columns(df)
    print(f"{len(df):,} instants | {len(feats)} features | "
          f"{df['eid'].nunique()} sessions | positive rate {df['y'].mean():.4f}\n")

    print("=" * 72)
    print("EXPERIMENT 1  Does the split rule change the conclusion?")
    print("=" * 72)
    leak = leakage_demonstration(df, feats)
    print(leak[["split", "f1", "precision", "recall", "pr_auc", "ece"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    print("\n" + "=" * 72)
    print("EXPERIMENT 2  In-domain versus out-of-domain")
    print("=" * 72)
    experiments = [
        ("tech", "4G", "5G"),
        ("tech", "5G", "4G"),
        ("mobility", "static", "mobile"),
        ("mobility", "mobile", "static"),
    ]
    parts = []
    for col, s_, t_ in experiments:
        r = domain_experiment(df, feats, col, s_, t_)
        parts.append(r)
        pd.concat(parts, ignore_index=True).to_csv(
            os.path.join(_HERE, "results_domain.csv"), index=False)
        print(f"  [done] {s_} -> {t_}", flush=True)
    all_res = pd.concat(parts, ignore_index=True)

    for exp, g in all_res.groupby("experiment", sort=False):
        print(f"\n--- {exp} ---")
        print(g[["model", "condition", "base_rate", "f1", "precision",
                 "recall", "pr_auc", "brier", "ece"]]
              .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    print("\n" + "=" * 72)
    print("GENERALIZATION GAP AND CALIBRATION DEGRADATION")
    print("=" * 72)
    gap = generalization_gap(all_res)
    print(gap[["experiment", "model", "f1_in", "f1_out", "f1_gap",
               "ece_in", "ece_out", "ece_degradation"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    all_res.to_csv(os.path.join(_HERE, "results_domain.csv"), index=False)
    gap.to_csv(os.path.join(_HERE, "results_gap.csv"), index=False)
    leak.to_csv(os.path.join(_HERE, "results_leakage.csv"), index=False)
    print("\nsaved: results_domain.csv, results_gap.csv, results_leakage.csv")


if __name__ == "__main__":
    main()


Writing models.py


## 4. Component 3/5 - calibration, shift, trust indicator

In [4]:
%%writefile trust.py
"""
AI4T Project 2 -- Cross-Domain Trust in 4G/5G YouTube QoE
Component 3 of 5: CALIBRATION, DISTRIBUTION SHIFT, TRUST INDICATOR

The brief requires:
  - the best model to be calibrated (Platt scaling / isotonic regression)
  - one simple measure of feature-distribution shift
  - a trust indicator combining model confidence, observed distribution shift
    and validation performance

Design note on the calibration split
------------------------------------
Calibrating on data the model was trained on produces a falsely good
calibration curve. We therefore split the SOURCE domain three ways by session:
    train (fit the model) | calib (fit the calibrator) | test (report)
No session appears in more than one of them.
"""

from __future__ import annotations

import json
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV

# scikit-learn >= 1.6 removed cv="prefit" in favour of FrozenEstimator.
# This shim keeps the code working on both old and new versions, which
# matters because Colab and local installs are often different releases.
try:
    from sklearn.frozen import FrozenEstimator

    def _calibrator(fitted_model, method):
        return CalibratedClassifierCV(FrozenEstimator(fitted_model), method=method)
except ImportError:                                   # scikit-learn < 1.6
    def _calibrator(fitted_model, method):
        return CalibratedClassifierCV(fitted_model, method=method, cv="prefit")
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

from models import (STATIC_CONTEXTS, add_persistence, evaluate,
                    expected_calibration_error, feature_columns, load_replayed)

warnings.filterwarnings("ignore")

_HERE = os.path.dirname(os.path.abspath(__file__))
RANDOM_STATE = 42


# --------------------------------------------------------------------------
# Distribution shift
# --------------------------------------------------------------------------

def population_stability_index(a: np.ndarray, b: np.ndarray, bins: int = 10) -> float:
    """PSI between a reference sample `a` and a new sample `b`.

    Chosen because it is the standard, easily explained shift measure in
    operational ML. Conventional reading:
        PSI < 0.10  negligible shift
        0.10-0.25   moderate shift
        PSI > 0.25  major shift -- model output should be treated with caution
    """
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    if len(a) < 10 or len(b) < 10:
        return np.nan
    edges = np.quantile(a, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    edges = np.unique(edges)
    if len(edges) < 3:
        return np.nan
    pa = np.histogram(a, bins=edges)[0].astype(float)
    pb = np.histogram(b, bins=edges)[0].astype(float)
    pa = np.clip(pa / pa.sum(), 1e-6, None)
    pb = np.clip(pb / pb.sum(), 1e-6, None)
    return float(np.sum((pb - pa) * np.log(pb / pa)))


def shift_report(src: pd.DataFrame, tgt: pd.DataFrame,
                 feats: list[str]) -> pd.DataFrame:
    """Per-feature PSI, source vs target domain."""
    rows = [{"feature": f,
             "psi": population_stability_index(src[f].values.astype(float),
                                               tgt[f].values.astype(float))}
            for f in feats]
    out = pd.DataFrame(rows).dropna().sort_values("psi", ascending=False)
    return out.reset_index(drop=True)


# --------------------------------------------------------------------------
# Trust indicator
# --------------------------------------------------------------------------

def trust_score(confidence: float, mean_psi: float, val_f1: float) -> float:
    """Combine the three ingredients the brief names, into [0, 1].

    trust = confidence_component * shift_penalty * validation_component

    - confidence_component : how far the calibrated probability is from the
      0.5 decision boundary, rescaled to [0,1]. A prediction sitting on the
      boundary carries little information.
    - shift_penalty        : exp(-PSI / 0.25). Equals 1 when the deployment
      distribution matches training, and decays as it diverges. 0.25 is the
      conventional "major shift" threshold, so a major shift costs ~63%.
    - validation_component : the model's measured F1 in the environment it was
      validated in. A model that never worked cannot be trusted anywhere.

    This is a decision aid, not a probability. The report must say so.
    """
    conf = abs(confidence - 0.5) * 2.0
    shift_penalty = float(np.exp(-max(mean_psi, 0.0) / 0.25))
    return float(np.clip(conf * shift_penalty * max(val_f1, 0.0), 0.0, 1.0))


def trust_band(score: float) -> str:
    if score >= 0.50:
        return "TRUST"
    if score >= 0.20:
        return "VERIFY"
    return "REJECT"


# --------------------------------------------------------------------------
# Calibration experiment
# --------------------------------------------------------------------------

def three_way_session_split(df: pd.DataFrame, seed: int = RANDOM_STATE):
    """Split a domain into train / calib / test by SESSION."""
    g = df["eid"].values
    idx_all = np.arange(len(df))
    s1 = GroupShuffleSplit(n_splits=1, test_size=0.40, random_state=seed)
    tr, rest = next(s1.split(idx_all, groups=g))
    rest_g = g[rest]
    s2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    ca_rel, te_rel = next(s2.split(rest, groups=rest_g))
    return tr, rest[ca_rel], rest[te_rel]


def build_base_models() -> dict:
    imp = lambda: SimpleImputer(strategy="median")
    return {
        "RandomForest": Pipeline([
            ("imp", imp()),
            ("clf", RandomForestClassifier(n_estimators=100, min_samples_leaf=20,
                                           class_weight="balanced_subsample",
                                           n_jobs=2, random_state=RANDOM_STATE)),
        ]),
        "GradientBoosting": HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.1, max_depth=6,
            class_weight="balanced", random_state=RANDOM_STATE),
    }


def calibration_experiment(df: pd.DataFrame, feats: list[str], column: str,
                           source: str, target: str) -> tuple:
    """Train on source, calibrate on source, evaluate in- and out-of-domain."""
    src = df[df[column] == source].reset_index(drop=True)
    tgt = df[df[column] == target].reset_index(drop=True)

    tr, ca, te = three_way_session_split(src)
    X = src[feats].values.astype("float32")
    y = src["y"].values
    Xt, yt = tgt[feats].values.astype("float32"), tgt["y"].values

    # distribution shift between the training environment and the target
    shift = shift_report(src.iloc[tr], tgt, feats)
    mean_psi = float(shift["psi"].mean())
    top_psi = shift.head(5)

    rows = []
    for name, model in build_base_models().items():
        model.fit(X[tr], y[tr])

        variants = {
            "uncalibrated": model,
            "platt": _calibrator(model, "sigmoid"),
            "isotonic": _calibrator(model, "isotonic"),
        }
        for vname, v in variants.items():
            if vname != "uncalibrated":
                v.fit(X[ca], y[ca])
            p_in = v.predict_proba(X[te])[:, 1]
            p_out = v.predict_proba(Xt)[:, 1]
            rows.append({"model": name, "calibration": vname,
                         "condition": "in-domain", **evaluate(y[te], p_in)})
            rows.append({"model": name, "calibration": vname,
                         "condition": "out-of-domain", **evaluate(yt, p_out)})

    res = pd.DataFrame(rows)
    res["experiment"] = f"{source} -> {target}"
    res["mean_psi"] = mean_psi
    return res, shift, top_psi, mean_psi


def main():
    df = add_persistence(load_replayed())
    df["mobility"] = np.where(df["context"].isin(STATIC_CONTEXTS), "static", "mobile")
    feats = feature_columns(df)

    all_res, shift_tables = [], {}
    for col, s, t in [("tech", "4G", "5G"), ("mobility", "mobile", "static")]:
        print(f"\n{'='*70}\nCALIBRATION: {s} -> {t}\n{'='*70}", flush=True)
        res, shift, top, mpsi = calibration_experiment(df, feats, col, s, t)
        all_res.append(res)
        shift_tables[f"{s}->{t}"] = shift
        print(f"mean PSI ({s} vs {t}) = {mpsi:.3f}")
        print("top-5 shifted features:")
        print(top.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
        print()
        print(res[["model", "calibration", "condition", "f1", "pr_auc",
                   "brier", "ece"]]
              .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    res = pd.concat(all_res, ignore_index=True)
    res.to_csv(os.path.join(_HERE, "results_calibration.csv"), index=False)
    for k, v in shift_tables.items():
        v.to_csv(os.path.join(_HERE,
                 f"results_shift_{k.replace('->','_to_')}.csv"), index=False)

    # ---- trust indicator applied to the headline transfer ----------------
    print(f"\n{'='*70}\nTRUST INDICATOR (4G-trained model)\n{'='*70}")
    hl = res[(res["experiment"] == "4G -> 5G") &
             (res["model"] == "RandomForest") &
             (res["calibration"] == "isotonic")]
    val_f1 = float(hl[hl["condition"] == "in-domain"]["f1"].iloc[0])
    mpsi = float(hl["mean_psi"].iloc[0])

    demo = []
    for conf in [0.95, 0.80, 0.65, 0.55]:
        for env, psi in [("in-domain 4G", 0.0), (f"out-of-domain 5G", mpsi)]:
            sc = trust_score(conf, psi, val_f1)
            demo.append({"confidence": conf, "environment": env,
                         "mean_psi": round(psi, 3), "val_f1": round(val_f1, 3),
                         "trust": round(sc, 3), "decision": trust_band(sc)})
    demo = pd.DataFrame(demo)
    print(demo.to_string(index=False))
    demo.to_csv(os.path.join(_HERE, "results_trust.csv"), index=False)
    print("\nsaved: results_calibration.csv, results_shift_*.csv, results_trust.csv")


if __name__ == "__main__":
    main()


Writing trust.py


## 5. Component 4/5 - grounded LLM explanation

In [5]:
%%writefile llm_explain.py
"""
AI4T Project 2 -- Cross-Domain Trust in 4G/5G YouTube QoE
Component 4 of 5: GROUNDED LLM EXPLANATION

The brief is explicit about the LLM's role:
  - it does NOT train the model and does NOT control the system
  - it receives STRUCTURED EVIDENCE produced by the model
  - it produces a short explanation stating the predicted event, likely causes,
    confidence/limitations, and what an operator should verify
  - students must verify the explanation invents no values absent from the
    evidence

This module therefore does three things:
  1. builds the evidence dictionary for a given instant of the replay
  2. sends it to a real LLM (API) or writes prompts for manual submission
  3. checks every number in the returned text against the evidence
"""

from __future__ import annotations

import json
import os
import re
import warnings

import numpy as np
import pandas as pd

from models import (STATIC_CONTEXTS, add_persistence, feature_columns,
                    load_replayed)
from trust import (_calibrator, build_base_models, three_way_session_split,
                   shift_report, trust_band, trust_score)

warnings.filterwarnings("ignore")
_HERE = os.path.dirname(os.path.abspath(__file__))
N_CASES = 20
RANDOM_STATE = 42


# --------------------------------------------------------------------------
# 1. Evidence construction
# --------------------------------------------------------------------------

def build_evidence(row: pd.Series, prob: float, mean_psi: float,
                   val_f1: float, in_domain: bool) -> dict:
    """Everything the LLM is allowed to know. Nothing else may appear
    in its explanation. Values are rounded so that string-matching the
    numbers back out of the generated text is reliable."""
    r = lambda v, n=1: (None if pd.isna(v) else round(float(v), n))
    ts = trust_score(prob, 0.0 if in_domain else mean_psi, val_f1)
    return {
        "session_id": row["eid"],
        "environment": {
            "technology": row["tech"],
            "context": row["context"],
            "matches_training_environment": bool(in_domain),
        },
        "prediction": {
            "event": "video stall within next 5 seconds",
            "probability": r(prob, 3),
            "threshold": 0.5,
            "predicted_positive": bool(prob >= 0.5),
        },
        "radio_evidence": {
            "rsrp_dbm_last": r(row.get("rsrp_last")),
            "rsrp_dbm_mean_10s": r(row.get("rsrp_mean")),
            "rsrp_trend_10s": r(row.get("rsrp_delta")),
            "rsrq_db_last": r(row.get("rsrq_last")),
            "snr_db_last": r(row.get("snr_last")),
            "cqi_last": r(row.get("cqi_last")),
        },
        "throughput_evidence": {
            "downlink_kbps_last": r(row.get("dl_bitrate_last")),
            "downlink_kbps_mean_10s": r(row.get("dl_bitrate_mean")),
            "fraction_of_zero_throughput_seconds": r(row.get("dl_zero_frac"), 2),
        },
        "session_evidence": {
            "elapsed_seconds": r(row.get("elapsed_s"), 0),
            "handovers_so_far": r(row.get("n_handovers"), 0),
            "stalls_so_far": r(row.get("n_stalls_so_far"), 0),
        },
        "reliability": {
            "model": "RandomForest + Platt scaling",
            "validation_f1_in_training_environment": r(val_f1, 3),
            "distribution_shift_psi": r(0.0 if in_domain else mean_psi, 3),
            "trust_score": r(ts, 3),
            "trust_band": trust_band(ts),
        },
    }


PROMPT_TEMPLATE = """You are assisting a mobile-network operator.

Below is structured evidence produced by a stall-prediction model. Write a
short explanation (maximum 120 words) containing exactly these four parts:
1. the predicted event,
2. the likely causes,
3. the confidence and its limitations,
4. what the operator should verify.

STRICT RULES:
- Use ONLY numbers that appear in the evidence below. Do not invent, estimate,
  round differently, or infer any value that is not present.
- Do not recommend changing the network. You explain; you do not act.
- If the trust_band is REJECT, say clearly that the prediction should not be
  used for an automatic decision.

EVIDENCE:
{evidence}
"""


def build_prompt(evidence: dict) -> str:
    return PROMPT_TEMPLATE.format(evidence=json.dumps(evidence, indent=2))


# --------------------------------------------------------------------------
# 2. Calling a real LLM (optional)
# --------------------------------------------------------------------------

def call_llm(prompt: str) -> str | None:
    """Try Anthropic, then OpenAI, using whichever API key is in the
    environment. Returns None if no key is configured, in which case the
    prompts are written out for manual submission instead."""
    if os.environ.get("ANTHROPIC_API_KEY"):
        try:
            import anthropic
            c = anthropic.Anthropic()
            m = c.messages.create(model="claude-sonnet-4-6", max_tokens=400,
                                  messages=[{"role": "user", "content": prompt}])
            return m.content[0].text.strip()
        except Exception as e:
            print("  anthropic call failed:", e)
    if os.environ.get("OPENAI_API_KEY"):
        try:
            from openai import OpenAI
            c = OpenAI()
            m = c.chat.completions.create(model="gpt-4o-mini", max_tokens=400,
                                          messages=[{"role": "user", "content": prompt}])
            return m.choices[0].message.content.strip()
        except Exception as e:
            print("  openai call failed:", e)
    return None


# --------------------------------------------------------------------------
# 3. Automated hallucination check
# --------------------------------------------------------------------------

def _numbers_in(obj) -> set:
    """Every numeric value present anywhere in the evidence."""
    out = set()
    if isinstance(obj, dict):
        for v in obj.values():
            out |= _numbers_in(v)
    elif isinstance(obj, list):
        for v in obj:
            out |= _numbers_in(v)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        out.add(round(float(obj), 3))
    return out


def check_grounding(explanation: str, evidence: dict) -> dict:
    """Extract every number from the generated text and test whether it is
    supported by the evidence. This produces the 'free of unsupported
    statements' judgement objectively instead of by eye.

    Small tolerances are allowed for legitimate restatement: an LLM may write
    '5 seconds' (the horizon) or convert 1500 kbps to 1.5 Mbps.
    """
    allowed = _numbers_in(evidence) | {5.0, 0.5, 120.0, 10.0, 100.0}
    allowed |= {round(v / 1000.0, 3) for v in allowed}   # kbps -> Mbps
    allowed |= {round(v * 100.0, 3) for v in allowed}    # fraction -> percent

    found = [float(m) for m in re.findall(r"-?\d+\.?\d*", explanation)]
    unsupported = []
    for f in found:
        if not any(abs(f - a) <= max(0.05, abs(a) * 0.02) for a in allowed):
            unsupported.append(f)

    return {
        "n_numbers_in_text": len(found),
        "n_unsupported": len(unsupported),
        "unsupported_values": unsupported,
        "auto_grounded": len(unsupported) == 0,
    }


def check_completeness(explanation: str) -> dict:
    """Does the explanation cover the four required parts?"""
    t = explanation.lower()
    return {
        "mentions_event": any(k in t for k in ["stall", "rebuffer", "buffering"]),
        "mentions_cause": any(k in t for k in ["because", "due to", "caused",
                                               "throughput", "rsrp", "signal",
                                               "snr", "cqi", "handover"]),
        "mentions_confidence": any(k in t for k in ["confidence", "probability",
                                                    "trust", "uncertain",
                                                    "reliab", "limitation"]),
        "mentions_verify": any(k in t for k in ["verify", "check", "inspect",
                                                "monitor", "confirm", "review"]),
    }


# --------------------------------------------------------------------------
# 4. Case generation
# --------------------------------------------------------------------------

def main():
    df = add_persistence(load_replayed())
    df["mobility"] = np.where(df["context"].isin(STATIC_CONTEXTS), "static", "mobile")
    feats = feature_columns(df)

    src = df[df["tech"] == "4G"].reset_index(drop=True)
    tgt = df[df["tech"] == "5G"].reset_index(drop=True)

    print("fitting the model that will produce the evidence...")
    tr, ca, te = three_way_session_split(src)
    X = src[feats].values.astype("float32"); y = src["y"].values
    rf = build_base_models()["RandomForest"]; rf.fit(X[tr], y[tr])
    cal = _calibrator(rf, "sigmoid"); cal.fit(X[ca], y[ca])

    from sklearn.metrics import f1_score
    p_te = cal.predict_proba(X[te])[:, 1]
    val_f1 = f1_score(y[te], (p_te >= 0.5).astype(int), zero_division=0)
    mean_psi = float(shift_report(src.iloc[tr], tgt, feats)["psi"].mean())
    print(f"  validation F1 = {val_f1:.3f} | mean PSI (4G vs 5G) = {mean_psi:.3f}")

    # Select 20 cases spanning the situations an operator actually meets:
    # in-domain positives and negatives, and out-of-domain positives and
    # negatives. A sample of only easy cases would not test the LLM.
    rng = np.random.RandomState(RANDOM_STATE)
    te_df = src.iloc[te].reset_index(drop=True)
    te_p = p_te
    p_tgt = cal.predict_proba(tgt[feats].values.astype("float32"))[:, 1]

    picks = []
    for name, frame, probs, indom in [("in-domain", te_df, te_p, True),
                                      ("out-of-domain", tgt, p_tgt, False)]:
        pos = np.where(frame["y"].values == 1)[0]
        neg = np.where(frame["y"].values == 0)[0]
        hi = np.argsort(-probs)[:200]
        for pool, k in [(pos, 3), (neg, 2), (hi, 5)]:
            if len(pool) == 0:
                continue
            sel = rng.choice(pool, size=min(k, len(pool)), replace=False)
            for i in sel:
                picks.append((name, frame.iloc[int(i)], float(probs[int(i)]), indom))

    picks = picks[:N_CASES]
    print(f"selected {len(picks)} cases")

    rows, prompts = [], []
    for i, (grp, row, prob, indom) in enumerate(picks, start=1):
        ev = build_evidence(row, prob, mean_psi, val_f1, indom)
        pr = build_prompt(ev)
        prompts.append(f"{'='*70}\nCASE {i:02d}  ({grp})\n{'='*70}\n{pr}\n")

        expl = call_llm(pr)
        rec = {
            "case_id": i,
            "group": grp,
            "session_id": row["eid"],
            "actual_outcome": int(row["y"]),
            "predicted_probability": round(prob, 3),
            "trust_band": ev["reliability"]["trust_band"],
            "evidence_json": json.dumps(ev),
            "explanation": expl if expl else "",
        }
        if expl:
            rec.update(check_grounding(expl, ev))
            rec.update(check_completeness(expl))
        # columns for the HUMAN judgement the brief requires
        rec.update({"human_factually_correct": "",
                    "human_complete": "",
                    "human_free_of_unsupported": "",
                    "human_notes": ""})
        rows.append(rec)

    cases = pd.DataFrame(rows)
    cases.to_csv(os.path.join(_HERE, "llm_cases.csv"), index=False)
    with open(os.path.join(_HERE, "llm_prompts.txt"), "w") as fh:
        fh.write("\n".join(prompts))

    got_llm = cases["explanation"].str.len().gt(0).sum()
    print(f"\nwrote llm_cases.csv ({len(cases)} cases) and llm_prompts.txt")
    if got_llm:
        print(f"{got_llm} explanations generated automatically")
        print(f"auto-grounded (no unsupported numbers): "
              f"{int(cases['auto_grounded'].sum())}/{got_llm}")
    else:
        print("\nNo API key found, so no explanations were generated.")
        print("Open llm_prompts.txt, paste each of the 20 prompts into any")
        print("chat LLM, and paste the replies into the 'explanation' column")
        print("of llm_cases.csv. Then re-run the grounding check on them.")
    print("\nFill in the three human_* columns by hand -- that is the")
    print("manual evaluation of 20 cases the brief requires.")


if __name__ == "__main__":
    main()


Writing llm_explain.py


## 6. Component 5/5 - figures

In [6]:
%%writefile figures.py
"""
AI4T Project 2 -- Cross-Domain Trust in 4G/5G YouTube QoE
Component 5 of 5: PUBLICATION FIGURES

Produces the six figures the brief requires:
  Fig 1  dataset / twin-state description
  Fig 2  baseline comparison (in-domain vs out-of-domain F1)
  Fig 3  leakage: random row split vs session-level split
  Fig 4  reliability diagram before and after calibration  [calibration result]
  Fig 5  feature-distribution shift (PSI)
  Fig 6  trust indicator decision map                      [limitation analysis]

All figures are saved at 300 dpi, greyscale-safe, with font sizes chosen for
a two-column IEEE page.
"""

from __future__ import annotations

import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from models import (STATIC_CONTEXTS, add_persistence, feature_columns,
                    load_replayed)
from trust import (_calibrator, build_base_models, three_way_session_split,
                   trust_band, trust_score)

warnings.filterwarnings("ignore")

_HERE = os.path.dirname(os.path.abspath(__file__))
FIGDIR = os.path.join(_HERE, "figures")
os.makedirs(FIGDIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True,
})
C = {"a": "#1f4e79", "b": "#c0504d", "c": "#7f7f7f", "d": "#4f81bd"}


def _save(fig, name):
    p = os.path.join(FIGDIR, name)
    fig.savefig(p)
    plt.close(fig)
    print(f"  saved {name}")


# --------------------------------------------------------------------------
def fig1_dataset(df):
    """Twin-state description: event rarity across the eight environments."""
    piv = df.pivot_table(index="tech", columns="context", values="y", aggfunc="mean") * 100
    cnt = df.pivot_table(index="tech", columns="context", values="eid", aggfunc="nunique")

    fig, ax = plt.subplots(figsize=(6.0, 2.4))
    im = ax.imshow(piv.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(piv.columns)), piv.columns)
    ax.set_yticks(range(len(piv.index)), piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.values[i,j]:.2f}%\n({cnt.values[i,j]} sess.)",
                    ha="center", va="center", fontsize=7,
                    color="white" if piv.values[i, j] > 4 else "black")
    ax.set_title("Stall rate within the 5 s prediction horizon, by environment")
    fig.colorbar(im, ax=ax, label="positive rate (%)")
    _save(fig, "fig1_dataset_description.png")


def fig2_baselines(gap):
    """In-domain vs out-of-domain F1 for every model and transfer."""
    exps = gap["experiment"].unique()
    fig, axes = plt.subplots(1, len(exps), figsize=(7.2, 2.6), sharey=True)
    for ax, e in zip(np.atleast_1d(axes), exps):
        g = gap[gap["experiment"] == e].sort_values("model")
        x = np.arange(len(g)); w = 0.38
        ax.bar(x - w/2, g["f1_in"], w, label="in-domain", color=C["a"])
        ax.bar(x + w/2, g["f1_out"], w, label="out-of-domain", color=C["b"])
        ax.set_xticks(x, [m[:4] for m in g["model"]], rotation=0)
        ax.set_title(e, fontsize=9)
        ax.set_ylim(0, 0.65)
    np.atleast_1d(axes)[0].set_ylabel("F1-score")
    np.atleast_1d(axes)[0].legend(loc="upper right")
    fig.suptitle("Model performance collapses across technology, not mobility", y=1.04)
    _save(fig, "fig2_baseline_comparison.png")


def fig3_leakage(leak):
    """The methodological control: split rule changes the headline number."""
    m = ["f1", "precision", "recall", "pr_auc"]
    lab = ["F1", "Precision", "Recall", "PR-AUC"]
    x = np.arange(len(m)); w = 0.38
    fig, ax = plt.subplots(figsize=(4.2, 2.6))
    ax.bar(x - w/2, leak.iloc[0][m].values, w, label="random row split", color=C["b"])
    ax.bar(x + w/2, leak.iloc[1][m].values, w, label="session-level split", color=C["a"])
    for i, k in enumerate(m):
        a, b = leak.iloc[0][k], leak.iloc[1][k]
        if b > 0:
            ax.text(i, max(a, b) + 0.02, f"+{100*(a-b)/b:.0f}%",
                    ha="center", fontsize=7, color=C["b"])
    ax.set_xticks(x, lab); ax.set_ylabel("score"); ax.set_ylim(0, 0.95)
    ax.legend(); ax.set_title("Row-level splitting inflates every metric")
    _save(fig, "fig3_leakage.png")


def fig4_reliability(df, feats):
    """Reliability diagrams: uncalibrated vs Platt, in- and out-of-domain."""
    src = df[df["tech"] == "4G"].reset_index(drop=True)
    tgt = df[df["tech"] == "5G"].reset_index(drop=True)
    tr, ca, te = three_way_session_split(src)
    X = src[feats].values.astype("float32"); y = src["y"].values
    Xt, yt = tgt[feats].values.astype("float32"), tgt["y"].values

    rf = build_base_models()["RandomForest"]
    rf.fit(X[tr], y[tr])
    cal = _calibrator(rf, "sigmoid"); cal.fit(X[ca], y[ca])

    def curve(model, Xe, ye, bins=10):
        p = model.predict_proba(Xe)[:, 1]
        edges = np.linspace(0, 1, bins + 1)
        xs, ys = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (p > lo) & (p <= hi)
            if m.sum() >= 20:
                xs.append(p[m].mean()); ys.append(ye[m].mean())
        return xs, ys

    fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.9), sharey=True)
    for ax, (Xe, ye, ttl) in zip(axes, [(X[te], y[te], "In-domain (4G)"),
                                        (Xt, yt, "Out-of-domain (5G)")]):
        ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect")
        xs, ys = curve(rf, Xe, ye);  ax.plot(xs, ys, "o-", color=C["b"], ms=3, label="uncalibrated")
        xs, ys = curve(cal, Xe, ye); ax.plot(xs, ys, "s-", color=C["a"], ms=3, label="Platt")
        ax.set_title(ttl); ax.set_xlabel("predicted probability")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    axes[0].set_ylabel("observed frequency"); axes[0].legend(loc="upper left")
    fig.suptitle("Calibration works in-domain but not under domain shift", y=1.03)
    _save(fig, "fig4_reliability.png")


def fig5_shift(shift):
    """Which variables actually move between 4G and 5G."""
    top = shift.head(12).iloc[::-1]
    fig, ax = plt.subplots(figsize=(4.4, 3.0))
    ax.barh(top["feature"], top["psi"], color=C["a"])
    for thr, lab, col in [(0.10, "moderate", C["c"]), (0.25, "major", C["b"])]:
        ax.axvline(thr, ls="--", lw=0.9, color=col)
        ax.text(thr, -0.8, lab, fontsize=7, color=col, ha="center")
    ax.set_xlabel("Population Stability Index (4G vs 5G)")
    ax.set_title("Radio-quality dynamics shift most between technologies")
    _save(fig, "fig5_distribution_shift.png")


def fig6_trust(mean_psi, val_f1):
    """Limitation analysis: identical confidence, opposite decision."""
    confs = np.linspace(0.5, 1.0, 120)
    psis = np.linspace(0.0, 0.6, 120)
    Z = np.array([[trust_score(c, p, val_f1) for c in confs] for p in psis])

    fig, ax = plt.subplots(figsize=(4.6, 3.0))
    im = ax.imshow(Z, origin="lower", aspect="auto", cmap="RdYlGn",
                   extent=[confs[0], confs[-1], psis[0], psis[-1]], vmin=0, vmax=0.6)
    cs = ax.contour(confs, psis, Z, levels=[0.20, 0.50], colors="k", linewidths=0.9)
    ax.clabel(cs, fmt={0.20: "REJECT/VERIFY", 0.50: "VERIFY/TRUST"}, fontsize=6)
    ax.axhline(0.0, color="k", lw=0.8)
    ax.axhline(mean_psi, color="k", lw=0.8, ls="--")
    ax.text(0.52, 0.02, "4G deployment (PSI=0)", fontsize=6.5)
    ax.text(0.52, mean_psi + 0.02, f"5G deployment (PSI={mean_psi:.2f})", fontsize=6.5)
    ax.plot([0.95, 0.95], [0.0, mean_psi], "ko-", ms=4, lw=1.2)
    ax.set_xlabel("model confidence"); ax.set_ylabel("distribution shift (mean PSI)")
    ax.set_title("Same 95% confidence, opposite operational decision")
    fig.colorbar(im, ax=ax, label="trust score")
    _save(fig, "fig6_trust_map.png")


# --------------------------------------------------------------------------
def main():
    print("loading results...")
    df = add_persistence(load_replayed())
    df["mobility"] = np.where(df["context"].isin(STATIC_CONTEXTS), "static", "mobile")
    feats = feature_columns(df)

    gap = pd.read_csv(os.path.join(_HERE, "results_gap.csv"))
    leak = pd.read_csv(os.path.join(_HERE, "results_leakage.csv"))
    shift = pd.read_csv(os.path.join(_HERE, "results_shift_4G_to_5G.csv"))
    cal = pd.read_csv(os.path.join(_HERE, "results_calibration.csv"))

    hl = cal[(cal["experiment"] == "4G -> 5G") & (cal["model"] == "RandomForest")
             & (cal["calibration"] == "isotonic")]
    val_f1 = float(hl[hl["condition"] == "in-domain"]["f1"].iloc[0])
    mean_psi = float(hl["mean_psi"].iloc[0])

    print("generating figures...")
    fig1_dataset(df)
    fig2_baselines(gap)
    fig3_leakage(leak)
    fig4_reliability(df, feats)
    fig5_shift(shift)
    fig6_trust(mean_psi, val_f1)
    print(f"\nall figures written to {FIGDIR}")


if __name__ == "__main__":
    main()


Writing figures.py


---
## 7. RUN: replay  *(~1 min)*  -> Table 1

In [7]:
!python twin_replay.py

loaded 262 sessions   skipped: {'no_player_events': 63, 'too_short': 2}
replayed 262 sessions -> 97,370 labelled instants
positive rate: 0.0263  (2,561 stall-imminent instants)

saved -> /content/replayed.pkl

stall rate by domain:
                  size      mean
tech context                    
4G   Indoor      12054  0.055832
     Mobility    16264  0.031972
     Outdoor     15800  0.016013
     Pedestrian  11080  0.067329
5G   Indoor       3325  0.003008
     Mobility    15734  0.013855
     Outdoor     10039  0.007471
     Pedestrian  13074  0.005048


## 8. RUN: models  *(~5 min, be patient)*  -> Tables 2, 3, 4

In [8]:
!python models.py

97,370 instants | 45 features | 262 sessions | positive rate 0.0263

EXPERIMENT 1  Does the split rule change the conclusion?
                        split    f1  precision  recall  pr_auc   ece
     random row split (leaky) 0.621      0.517   0.778   0.622 0.064
session-level split (correct) 0.604      0.570   0.642   0.455 0.058

EXPERIMENT 2  In-domain versus out-of-domain
  [done] 4G -> 5G
  [done] 5G -> 4G
  [done] static -> mobile
  [done] mobile -> static

--- 4G -> 5G ---
             model          condition  base_rate    f1  precision  recall  pr_auc  brier   ece
       Persistence     in-domain (4G)      0.056 0.431      0.441   0.421   0.218  0.062 0.030
       Persistence out-of-domain (5G)      0.009 0.085      0.088   0.081   0.015  0.015 0.007
LogisticRegression     in-domain (4G)      0.056 0.420      0.285   0.798   0.372  0.116 0.221
LogisticRegression out-of-domain (5G)      0.009 0.030      0.021   0.051   0.022  0.029 0.079
      RandomForest     in-domain (4G)   

## 9. RUN: calibration and trust  *(~4 min)*  -> Tables 5, 6

In [9]:
!python trust.py


CALIBRATION: 4G -> 5G
mean PSI (4G vs 5G) = 0.433
top-5 shifted features:
  feature   psi
rsrq_mean 2.405
 rsrq_min 2.166
 rsrq_max 2.145
rsrq_last 2.063
 rsrq_std 1.456

           model  calibration     condition    f1  pr_auc  brier   ece
    RandomForest uncalibrated     in-domain 0.534   0.438  0.035 0.068
    RandomForest uncalibrated out-of-domain 0.000   0.023  0.013 0.047
    RandomForest        platt     in-domain 0.518   0.438  0.028 0.021
    RandomForest        platt out-of-domain 0.000   0.023  0.009 0.009
    RandomForest     isotonic     in-domain 0.394   0.402  0.027 0.014
    RandomForest     isotonic out-of-domain 0.000   0.021  0.009 0.004
GradientBoosting uncalibrated     in-domain 0.476   0.380  0.033 0.023
GradientBoosting uncalibrated out-of-domain 0.005   0.019  0.009 0.007
GradientBoosting        platt     in-domain 0.296   0.380  0.027 0.026
GradientBoosting        platt out-of-domain 0.000   0.019  0.009 0.013
GradientBoosting     isotonic     in-domain 0.2

## 10. RUN: figures  *(~2 min)*  -> Figures 1-6

In [10]:
!python figures.py

loading results...
generating figures...
  saved fig1_dataset_description.png
  saved fig2_baseline_comparison.png
  saved fig3_leakage.png
  saved fig4_reliability.png
  saved fig5_distribution_shift.png
  saved fig6_trust_map.png

all figures written to /content/figures


## 11. RUN: LLM evidence and prompts  *(~1 min)*

**Optional:** paste an API key in the cell below to generate the 20
explanations automatically. Leave it blank and the notebook writes
`llm_prompts.txt` instead, for pasting into any chat LLM by hand.

In [11]:
import os
os.environ['ANTHROPIC_API_KEY'] = ''   # optional
os.environ['OPENAI_API_KEY']    = ''   # optional
!python llm_explain.py

fitting the model that will produce the evidence...
  validation F1 = 0.518 | mean PSI (4G vs 5G) = 0.433
selected 20 cases

wrote llm_cases.csv (20 cases) and llm_prompts.txt

No API key found, so no explanations were generated.
Open llm_prompts.txt, paste each of the 20 prompts into any
chat LLM, and paste the replies into the 'explanation' column
of llm_cases.csv. Then re-run the grounding check on them.

Fill in the three human_* columns by hand -- that is the
manual evaluation of 20 cases the brief requires.


## 12. Check everything was produced

In [12]:
import pandas as pd, os
print('--- figures ---'); print(os.listdir('figures'))
print('--- result tables ---')
for f in sorted(f for f in os.listdir('.') if f.endswith('.csv')): print(' ', f)
print()
display(pd.read_csv('results_gap.csv').round(3))
display(pd.read_csv('results_trust.csv'))

--- figures ---
['fig2_baseline_comparison.png', 'fig1_dataset_description.png', 'fig4_reliability.png', 'fig5_distribution_shift.png', 'fig6_trust_map.png', 'fig3_leakage.png']
--- result tables ---
  llm_cases.csv
  results_calibration.csv
  results_domain.csv
  results_gap.csv
  results_leakage.csv
  results_shift_4G_to_5G.csv
  results_shift_mobile_to_static.csv
  results_trust.csv



,experiment,model,f1_in,f1_out,f1_gap,ece_in,ece_out,ece_degradation,brier_in,brier_out,base_rate_in,base_rate_out
0,4G -> 5G,GradientBoosting,0.345,0.000,0.345,0.032,0.007,-0.025,0.049,0.009,0.056,0.009
1,4G -> 5G,LogisticRegression,0.420,0.030,0.390,0.221,0.079,-0.142,0.116,0.029,0.056,0.009
2,4G -> 5G,Persistence,0.431,0.085,0.346,0.030,0.007,-0.022,0.062,0.015,0.056,0.009
3,4G -> 5G,RandomForest,0.561,0.000,0.561,0.071,0.046,-0.026,0.048,0.013,0.056,0.009
4,5G -> 4G,GradientBoosting,0.091,0.012,0.079,0.027,0.050,0.022,0.018,0.047,0.011,0.040
5,5G -> 4G,LogisticRegression,0.040,0.108,-0.068,0.381,0.320,-0.062,0.209,0.194,0.011,0.040
6,5G -> 4G,Persistence,0.057,0.413,-0.357,0.009,0.021,0.013,0.019,0.045,0.011,0.040
7,5G -> 4G,RandomForest,0.029,0.000,0.029,0.038,0.021,-0.017,0.014,0.041,0.011,0.040
8,mobile -> static,GradientBoosting,0.453,0.292,0.162,0.044,0.045,0.001,0.038,0.035,0.033,0.025
9,mobile -> static,LogisticRegression,0.210,0.199,0.011,0.247,0.228,-0.020,0.132,0.109,0.033,0.025


,confidence,environment,mean_psi,val_f1,trust,decision
0,0.95,in-domain 4G,0.000,0.394,0.354,VERIFY
1,0.95,out-of-domain 5G,0.433,0.394,0.063,REJECT
2,0.80,in-domain 4G,0.000,0.394,0.236,VERIFY
3,0.80,out-of-domain 5G,0.433,0.394,0.042,REJECT
4,0.65,in-domain 4G,0.000,0.394,0.118,REJECT
5,0.65,out-of-domain 5G,0.433,0.394,0.021,REJECT
6,0.55,in-domain 4G,0.000,0.394,0.039,REJECT
7,0.55,out-of-domain 5G,0.433,0.394,0.007,REJECT


## 13. Download everything for your GitHub repo

Creates one zip containing the five `.py` files, all result CSVs and all
six figures. Download it, unzip it, commit it.

In [13]:
!zip -r ai4t_project2_submission.zip *.py *.csv figures/ llm_prompts.txt -q
from google.colab import files
files.download('ai4t_project2_submission.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>